In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import BertTokenizer

In [2]:
import pandas as pd

# Đọc file CSV
df = pd.read_csv('/kaggle/input/dataset-capec-csv/dataset_capec.csv') # Đọc file CSV
print(df.head())

                                                text         label
0  GET /blog/index.php/2020/04/04/voluptatum-repr...  000 - Normal
1                           GET /blog/xmlrpc.php?rsd  000 - Normal
2  GET /blog/index.php/2020/04/04/nihil-tenetur-e...  000 - Normal
3  GET /blog/index.php/2020/04/04/explicabo-qui-f...  000 - Normal
4  GET /blog/index.php/2020/04/04/explicabo-qui-f...  000 - Normal


In [3]:
# Optional (not effect very much)
df['text'] = df['text'].str.replace('/',' ')
df.head()

,text,label
0,GET blog index.php 2020 04 04 voluptatum-repr...,000 - Normal
1,GET blog xmlrpc.php?rsd,000 - Normal
2,GET blog index.php 2020 04 04 nihil-tenetur-e...,000 - Normal
3,GET blog index.php 2020 04 04 explicabo-qui-f...,000 - Normal
4,GET blog index.php 2020 04 04 explicabo-qui-f...,000 - Normal


In [4]:
# Đếm số lượng record cho mỗi loại label
label_counts = df['label'].value_counts()

# Lọc các label có số lượng record > 20000
labels_above_20000 = label_counts[label_counts > 10000].index

# Lấy 20,000 record đầu tiên cho các label đó
df_above_20000 = df[df['label'].isin(labels_above_20000)]
df_above_20000 = df_above_20000.groupby('label').head(10000)

# Lấy các record còn lại (cho các label không có số lượng lớn hơn 20000)
df_below_20000 = df[~df['label'].isin(labels_above_20000)]

# Ghép các record lại với nhau
df_combined = pd.concat([df_below_20000, df_above_20000])

# Xáo trộn dữ liệu sau khi ghép
df_combined = df_combined.sample(frac=1).reset_index(drop=True)
df = df_combined

print(df_combined)

                                                    text  \
0      GET  blog index.php wp-json oembed 1.0 embed?u...   
1      GET  blog%0Acat+%2Fetc%2Fpasswd%0A index.php 2...   
2      GET  blog index.php any%3F%0ASet-cookie%3A+Tam...   
3      GET  blog index.php%0Acat+%2Fetc%2Fpasswd%0A q...   
4      GET  blog wp-content %3Bprint%28chr%28122%29.c...   
...                                                  ...   
70688  GET  blog https%3A%2F%2Fwww.google.com%2F 2020...   
70689  GET  blog %27%3Bprint%28chr%28122%29.chr%2897%...   
70690  GET  blog %28 2020 04 04 consequatur-ad-error-...   
70691  GET  blog www.google.com%2Fsearch%3Fq%3DOWASP%...   
70692  GET  blog wp-content%0Acat+%2Fetc%2Fpasswd%0A ...   

                               label  
0                       000 - Normal  
1       34 - HTTP Response Splitting  
2       34 - HTTP Response Splitting  
3       34 - HTTP Response Splitting  
4               242 - Code Injection  
...                              ...  
70688 

In [4]:

# Phân chia dữ liệu train-test
X_train, X_test, y_train, y_test = train_test_split(df['text'], 
                                                    df['label'], 
                                                    test_size=0.2, 
                                                    shuffle=True) # shuffle=True

In [5]:
# Encode labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)  # Encode train labels
y_test_encoded = label_encoder.transform(y_test)        # Encode test labels

# In danh sách nhãn đã mã hóa
print(label_encoder.classes_)  # ['000 - Normal', '001 - XSS', ...]
num_classes = len(label_encoder.classes_)  # Số lượng lớp
num_classes

['000 - Normal' '126 - Path Traversal' '153 - Input Data Manipulation'
 '194 - Fake the Source of Data' '242 - Code Injection'
 '272 - Protocol Manipulation' '310 - Scanning for Vulnerable Software'
 '34 - HTTP Response Splitting' '66 - SQL Injection']


9

In [6]:
# Tải Tokenizer của SecBERT
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("jackaduma/SecBERT")  # SecBERT

vocab.txt:   0%|          | 0.00/378k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [7]:
# Token hóa văn bản
# Tokenize dữ liệu train và test
def preprocess_data(texts, tokenizer, max_length=90):
    # Convert the Pandas Series to a list of strings
    texts = texts.tolist()  # This line is added to fix the error
    return tokenizer(
        texts,
        padding='max_length',        # Thêm padding
        truncation=True,             # Cắt ngắn văn bản
        max_length=max_length,       # Độ dài tối đa
        return_tensors='tf'          # Trả về Tensor
    )

X_train_tokens = preprocess_data(X_train, tokenizer)
X_test_tokens = preprocess_data(X_test, tokenizer)

In [8]:
from transformers import TFBertModel
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.models import Model

# Tải mô hình SecBERT
secbert_model = TFBertModel.from_pretrained("jackaduma/SecBERT")

# Đóng băng các lớp SecBERT (tùy chọn, có thể fine-tune)
secbert_model.trainable = False

model.safetensors:   0%|          | 0.00/336M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.dense.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias', 'bert.embeddings.position_ids', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions without further training.


In [9]:
from tensorflow.keras.layers import Lambda

secbert_model.trainable = True

# Input layers
input_ids = tf.keras.Input(shape=(90,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(90,), dtype=tf.int32, name="attention_mask")

# Gọi SecBERT qua Lambda layer, specify output_shape
bert_outputs = Lambda(
    lambda x: secbert_model(input_ids=x[0], attention_mask=x[1])[0][:, 0, :],
    output_shape=(768,)  # Specify the output shape here
)([input_ids, attention_mask])

# Thêm các lớp Dense
x = Dense(256, activation="relu")(bert_outputs)
x = Dropout(0.25)(x)
output = Dense(num_classes, activation="softmax")(x)

# Tạo mô hình
model = Model(inputs=[input_ids, attention_mask], outputs=output)

# Compile mô hình
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), 
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])

In [10]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    {
        "input_ids": X_train_tokens["input_ids"],
        "attention_mask": X_train_tokens["attention_mask"]
    },
    y_train_encoded,
    validation_split=0.2,
    epochs=15,
    batch_size=128,
    callbacks=[early_stopping]
)


Epoch 1/15
2959/2959 ━━━━━━━━━━━━━━━━━━━━ 1161s 388ms/step - accuracy: 0.5059 - loss: 1.3726 - val_accuracy: 0.5740 - val_loss: 1.1143
Epoch 2/15
2959/2959 ━━━━━━━━━━━━━━━━━━━━ 1141s 386ms/step - accuracy: 0.5659 - loss: 1.1382 - val_accuracy: 0.5822 - val_loss: 1.0888
Epoch 3/15
2959/2959 ━━━━━━━━━━━━━━━━━━━━ 1140s 385ms/step - accuracy: 0.5708 - loss: 1.1153 - val_accuracy: 0.5839 - val_loss: 1.0780
Epoch 4/15
2959/2959 ━━━━━━━━━━━━━━━━━━━━ 1141s 386ms/step - accuracy: 0.5745 - loss: 1.1005 - val_accuracy: 0.5836 - val_loss: 1.0690
Epoch 5/15
2959/2959 ━━━━━━━━━━━━━━━━━━━━ 1142s 386ms/step - accuracy: 0.5747 - loss: 1.0925 - val_accuracy: 0.5844 - val_loss: 1.0636
Epoch 6/15
2959/2959 ━━━━━━━━━━━━━━━━━━━━ 1141s 386ms/step - accuracy: 0.5744 - loss: 1.0871 - val_accuracy: 0.5839 - val_loss: 1.0589
Epoch 7/15
2959/2959 ━━━━━━━━━━━━━━━━━━━━ 1143s 386ms/step - accuracy: 0.5759 - loss: 1.0805 - val_accuracy: 0.5842 - val_loss: 1.0555
Epoch 8/15
2959/2959 ━━━━━━━━━━━━━━━━━━━━ 1140s 385ms/s

In [11]:
from sklearn.metrics import classification_report

# Dự đoán trên tập validation hoặc test
y_pred = model.predict({
    "input_ids": X_test_tokens["input_ids"],
    "attention_mask": X_test_tokens["attention_mask"]
})

# Lấy nhãn dự đoán từ xác suất (với softmax)
y_pred_labels = y_pred.argmax(axis=1)

label_names = label_encoder.classes_  # Lấy danh sách tên nhãn

# Tính toán precision, recall, f1-score
print(classification_report(y_test_encoded, y_pred_labels, target_names=label_names))

3699/3699 ━━━━━━━━━━━━━━━━━━━━ 321s 86ms/step
                                        precision    recall  f1-score   support

                          000 - Normal       0.59      0.72      0.65     45334
                  126 - Path Traversal       0.66      0.46      0.54      3502
         153 - Input Data Manipulation       0.00      0.00      0.00       270
         194 - Fake the Source of Data       0.00      0.00      0.00     11297
                  242 - Code Injection       0.00      0.00      0.00      2775
           272 - Protocol Manipulation       0.00      0.00      0.00      1380
310 - Scanning for Vulnerable Software       0.95      0.05      0.09       444
          34 - HTTP Response Splitting       0.00      0.00      0.00      3831
                    66 - SQL Injection       0.57      0.70      0.63     49527

                              accuracy                           0.58    118360
                             macro avg       0.31      0.21      0.21   

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
